- 001 hello world
- 002 add non-absolute-value version of daysdiff. should reall rename the other "absdaysdiff" but it propagate throughout the code base. also, it doesn't crash now
- 003 shrink to papers with DOIs
- 004 include hand-checked PPPs so model does not have missing values

In [ ]:
from ppp_paths import databulk, dataraw, dataint
maxpatentpaperdaysdiff = 730
pqrconfthres = 0.9


In [ ]:
import pandas as pd
import numpy as np
import gc

Get the patent-to-paper SELF citations, regardless of temporal gap

In [ ]:
dtypes = {
    'magid': 'str', 
    'patent_id': 'str'
}
pcspairs = pd.read_csv(databulk + '_pcs_oa.csv', dtype=dtypes)
pcspairs = pcspairs[pcspairs['uspto']==1]
pcspairs = pcspairs[pcspairs['confscore']>3]
pcspairs = pcspairs[pcspairs['self']=='isself']
pcspairs['patent'] = pcspairs['patent'].str.removeprefix("us-")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b1")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b2")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a1")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a2")
pcspairs.rename(columns={'oaid': 'magid'}, inplace=True)
pcspairs.rename(columns={'patent': 'patent_id'}, inplace=True)
pcspairs = pcspairs[['magid', 'patent_id']]
pcspairs['magid'] = pcspairs['magid'].astype(str)
pcspairs.drop_duplicates(inplace=True)
pcspairs.dropna(inplace=True)
pcspairs['selfcite'] = True
pcspairs

Get PQR paper/patent combinations within N years (can't look at all of them)

In [ ]:
dtypes = {
    'work_id': 'str', 
    'author_id': 'str'
}
pqrpapers = pd.read_csv(databulk + 'pqrs_authorid_workid.tsv', sep='\t', dtype=dtypes)
dtypes = {
    'patent_id': 'str', 
    'inventor_id': 'str'
}
pqrpatents = pd.read_csv(databulk + 'inventorid_patentid.tsv', sep='\t')
dtypes = {
    'author_id': 'str', 
    'inventor_id': 'str'
}
pqrs = pd.read_csv(databulk + 'pqrs_dataset.tsv', sep='\t', dtype=dtypes)
pqrs = pqrs[['author_id', 'inventor_id', 'confidence']].drop_duplicates()
pqrs = pqrs[pqrs['confidence'] >= pqrconfthres]
pqrpairs = pqrs.merge(pqrpapers, on='author_id', how='left')
del pqrpapers
pqrpairs = pqrpairs.merge(pqrpatents, on='inventor_id', how='left')
del pqrpatents
pqrpairs = pqrpairs[['work_id', 'patent_id']].drop_duplicates()
pqrpairs.rename(columns={'work_id': 'magid'}, inplace=True)
pqrpairs['magid'] = pqrpairs['magid'].astype(str)

pqrpairs['patent_id'] = pqrpairs['patent_id'].str.replace('US-', '', regex=False)
print(pqrpairs)
gc.collect()

In [ ]:
df_handcoded_josh = pd.read_csv(dataraw + 'Check Pairs_Josh.csv')
# df_handcoded_henry = pd.read_csv(dataraw + 'mag_patent_ppp_Henry.csv', encoding='unicode_escape')
df_handcoded_josh = df_handcoded_josh.rename(columns={'patent': 'patent_id'})
df_handcoded_josh = df_handcoded_josh[['magid', 'patent_id']]
df_handcoded_josh['patent_id'] = df_handcoded_josh['patent_id'].str.removeprefix('US-')
df_handcoded_josh['magid'] = df_handcoded_josh['magid'].astype(str)
df_handcoded_josh['patent_id'] = df_handcoded_josh['patent_id'].astype(str)
df_handcoded_henry = pd.read_csv(dataraw + 'mag_patent_ppp_Henry.csv', encoding='unicode_escape')
df_handcoded_henry = df_handcoded_henry.rename(columns={'Magid': 'magid', 'Patent': 'patent_id'})
df_handcoded_henry = df_handcoded_henry[['magid', 'patent_id']]
df_handcoded_henry['patent_id'] = df_handcoded_henry['patent_id'].str.removeprefix('US-')
df_handcoded_henry['magid'] = df_handcoded_henry['magid'].astype(str)
df_handcoded_henry['patent_id'] = df_handcoded_henry['patent_id'].astype(str)
df_handcoded_cornellphds = pd.read_csv(dataraw + "cornell_phd_audit_of_pairs.csv", encoding='unicode_escape')
df_handcoded_cornellphds.columns.values[0] = 'magid'
df_handcoded_cornellphds.columns.values[3] = 'patent_id'
# df_handcoded_cornellphds = df_handcoded_cornellphds[['magid', 'patent_id', 'response1', 'response2']]
df_handcoded_cornellphds = df_handcoded_cornellphds[['magid', 'patent_id']]
df_handcoded_cornellphds['patent_id'] = df_handcoded_cornellphds['patent_id'].str.removeprefix('US-')
df_handcoded_cornellphds['magid'] = df_handcoded_cornellphds['magid'].astype(str)
df_handcoded_cornellphds['patent_id'] = df_handcoded_cornellphds['patent_id'].astype(str)
df_handcoded_cornellphds = df_handcoded_cornellphds.drop_duplicates(subset=['magid', 'patent_id'])



In [ ]:
df_handcoded_josh
df_handcoded_cornellphds
df_handcoded_henry
df_handcoded_henry.to_csv(dataint + 'df_handcoded_henry.csv')

In [ ]:
ppphandcheck = pd.concat([df_handcoded_josh, df_handcoded_henry, df_handcoded_cornellphds])
ppphandcheck = ppphandcheck.drop_duplicates()
ppphandcheck = ppphandcheck.dropna()
ppphandcheck

In [ ]:
potentialpairs = pd.concat([pcspairs, pqrpairs, ppphandcheck], ignore_index=True)
# del pcspairs 
# del pqrpairs
gc.collect()
potentialpairs['selfcite'].fillna(False, inplace=True)
potentialpairs.dropna(inplace=True)
potentialpairs = potentialpairs.sort_values('selfcite', ascending=False)
potentialpairs.drop_duplicates(subset=['magid', 'patent_id'], keep='first', inplace=True)
potentialpairs

In [ ]:
dtypes = {
    'paperid': 'str'              # magid
}
# paperyears = pd.read_csv(databulk + 'paperdates.tsv.zip', sep='\t', header=None, compression='zip')  
paperdates = pd.read_csv(databulk + 'paperdates.zip', sep='\t', dtype=dtypes, parse_dates=[0], date_format='%Y%m%d', header=None, compression='zip')  
# paperdates = pd.read_csv(databulk + 'paperdates.tsv', sep='\t', dtype=dtypes, parse_dates=[0], date_format='%Y%m%d', header=None)  
# paperdates.rename(columns={1: 'paperpubdate', 0: 'magid'}, inplace=True) 
paperdates = paperdates.rename(columns={0: 'magid'})
paperdates = paperdates.rename(columns={1: 'paperpubdate'})
paperdates['magid'] = paperdates['magid'].astype(str)
paperdates.dropna(inplace=True)
paperdates.drop_duplicates(inplace=True)
paperdates                        

In [ ]:
dtypes = {
    'patent_id': 'str'              # magid
}
patentdates = pd.read_csv(databulk + 'g_application.tsv.zip', sep='\t', dtype=dtypes, parse_dates=['filing_date'], date_format='%Y%m%d', compression='zip')
patentdates = patentdates[['patent_id', 'filing_date']].drop_duplicates()
# patentdates = patentdates.dropna(inplace=True)
patentdates

In [ ]:
patentdates
paperdates
potentialpairs
temp = potentialpairs[potentialpairs['selfcite'] == 1]
temp

In [ ]:
pcspairsdates = pcspairs.merge(paperdates, on='magid', how='left')
pcspairsdates

In [ ]:
potentialpairsplusdates = potentialpairs.merge(paperdates, on='magid', how='left')
# del paperdates
potentialpairsplusdates = potentialpairsplusdates.merge(patentdates, on='patent_id', how='left')  
# del patentdates
potentialpairsplusdates.dropna(inplace=True)
gc.collect()
potentialpairsplusdates

In [ ]:
temp = potentialpairsplusdates[potentialpairsplusdates['selfcite'] == True]
temp
 
# temp.value_counts('paperpubdate') 

In [ ]:
potentialpairsplusdates.dropna(inplace=True)
potentialpairsplusdates


In [ ]:
# potentialpairsplusdates.value_counts('selfcite')
potentialpairsplusdates['filing_date'] = pd.to_datetime(potentialpairsplusdates['filing_date'], errors='coerce')
potentialpairsplusdates['paperpubdate'] = pd.to_datetime(potentialpairsplusdates['paperpubdate'], errors='coerce')

In [ ]:
potentialpairsplusdates['daysdiffnoabs'] = (potentialpairsplusdates['filing_date'] - potentialpairsplusdates['paperpubdate']).dt.days
potentialpairsplusdates['daysdiff'] = abs(potentialpairsplusdates['filing_date'] - potentialpairsplusdates['paperpubdate']).dt.days
print(potentialpairsplusdates['daysdiff'].value_counts())


keep the pqr pairs within 2 years of each other, but all selfcite pairs (we will use temporally separated selfcites as negative training examples)

In [ ]:
potentialpairswithinwindow = potentialpairsplusdates[(potentialpairsplusdates['selfcite'] == 1) | (potentialpairsplusdates['daysdiff'] < maxpatentpaperdaysdiff)]
potentialpairswithinwindow = potentialpairswithinwindow[['magid', 'patent_id', 'daysdiff', 'daysdiffnoabs', 'selfcite']]
potentialpairswithinwindow['magid'] = potentialpairswithinwindow['magid'].astype(str)
potentialpairswithinwindow['patent_id'] = potentialpairswithinwindow['patent_id'].astype(str)
potentialpairswithinwindow

In [ ]:
# potentialpairswithinwindow = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
test = pd.read_csv(dataint + 'df_handcoded_henry.csv', dtype=str)
print(test.shape)
test2 = pd.merge(test, potentialpairswithinwindow, on=['magid', 'patent_id'], how='inner')
print(test2.shape)


In [ ]:
pqrdoi = pd.read_csv(databulk + 'paperdoi.zip', sep='\t', dtype='str', compression='zip')
pqrdoi = pqrdoi.rename(columns={'paperid':'magid'})
pqrdoi

In [ ]:
print(potentialpairswithinwindow.shape)
potentialpairswithinwindow = pd.merge(potentialpairswithinwindow, pqrdoi, on='magid', how='inner')
print(potentialpairswithinwindow.shape)

In [ ]:
#15015162
missing_doi = potentialpairswithinwindow['doi'].isnull().sum()
print(f"Number of rows missing 'doi': {missing_doi}")
potentialpairswithinwindow = potentialpairswithinwindow.dropna()
potentialpairswithinwindow

In [ ]:
potentialpairswithinwindow.to_parquet(dataint + 'potentialpairswithinwindow.parquet', compression='snappy', index=False)
potentialpairswithinwindow.to_csv(dataint + 'potentialpairswithinwindow.tsv', sep='\t', index=False)

In [ ]:
potentialpairswithinwindow = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
test = pd.read_csv(dataint + 'df_handcoded_henry.csv', dtype=str)
print(test.shape)
test2 = pd.merge(test, potentialpairswithinwindow, on=['magid', 'patent_id'], how='inner')
print(test2.shape)
